In [17]:
from pathlib import Path

file_path = Path("marketing_campaign.csv")

print("EXISTS:", file_path.exists())
print("SIZE_BYTES:", file_path.stat().st_size if file_path.exists() else "NO_FILE")

with open(file_path, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline()
        print(f"LINE_{i+1}:", repr(line[:300]))

EXISTS: True
SIZE_BYTES: 220188
LINE_1: 'ID\tYear_Birth\tEducation\tMarital_Status\tIncome\tKidhome\tTeenhome\tDt_Customer\tRecency\tMntWines\tMntFruits\tMntMeatProducts\tMntFishProducts\tMntSweetProducts\tMntGoldProds\tNumDealsPurchases\tNumWebPurchases\tNumCatalogPurchases\tNumStorePurchases\tNumWebVisitsMonth\tAcceptedCmp3\tAcceptedCmp4\tAcceptedCmp5\tAccepte'
LINE_2: '5524\t1957\tGraduation\tSingle\t58138\t0\t0\t04-09-2012\t58\t635\t88\t546\t172\t88\t88\t3\t8\t10\t4\t7\t0\t0\t0\t0\t0\t0\t3\t11\t1\n'
LINE_3: '2174\t1954\tGraduation\tSingle\t46344\t1\t1\t08-03-2014\t38\t11\t1\t6\t2\t1\t6\t2\t1\t1\t2\t5\t0\t0\t0\t0\t0\t0\t3\t11\t0\n'
LINE_4: '4141\t1965\tGraduation\tTogether\t71613\t0\t0\t21-08-2013\t26\t426\t49\t127\t111\t21\t42\t1\t8\t2\t10\t4\t0\t0\t0\t0\t0\t0\t3\t11\t0\n'
LINE_5: '6182\t1984\tGraduation\tTogether\t26646\t1\t0\t10-02-2014\t26\t11\t4\t20\t10\t3\t5\t2\t2\t0\t4\t6\t0\t0\t0\t0\t0\t0\t3\t11\t0\n'


In [18]:
import polars as pl

df = pl.read_csv("marketing_campaign.csv", separator="\t")

print("SHAPE:", df.shape)
print("COLUMN_COUNT:", len(df.columns))
print("COLUMNS:", df.columns)
print("\nHEAD:")
print(df.head(3))

SHAPE: (2240, 29)
COLUMN_COUNT: 29
COLUMNS: ['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']

HEAD:
shape: (3, 29)
┌──────┬────────────┬────────────┬─────────────┬───┬──────────┬─────────────┬───────────┬──────────┐
│ ID   ┆ Year_Birth ┆ Education  ┆ Marital_Sta ┆ … ┆ Complain ┆ Z_CostConta ┆ Z_Revenue ┆ Response │
│ ---  ┆ ---        ┆ ---        ┆ tus         ┆   ┆ ---      ┆ ct          ┆ ---       ┆ ---      │
│ i64  ┆ i64        ┆ str        ┆ ---         ┆   ┆ i64      ┆ ---         ┆ i64       ┆ i64      │
│      ┆            ┆            ┆ str         ┆   ┆          ┆ i64         ┆    

In [19]:
print("DTYPES:")
for col, dtype in zip(df.columns, df.dtypes):
    print(f"{col}: {dtype}")

print("\nNULL_COUNT:")
print(df.null_count())

duplicate_rows = df.is_duplicated().sum()
print("\nDUPLICATE_ROWS:", duplicate_rows)

DTYPES:
ID: Int64
Year_Birth: Int64
Education: String
Marital_Status: String
Income: Int64
Kidhome: Int64
Teenhome: Int64
Dt_Customer: String
Recency: Int64
MntWines: Int64
MntFruits: Int64
MntMeatProducts: Int64
MntFishProducts: Int64
MntSweetProducts: Int64
MntGoldProds: Int64
NumDealsPurchases: Int64
NumWebPurchases: Int64
NumCatalogPurchases: Int64
NumStorePurchases: Int64
NumWebVisitsMonth: Int64
AcceptedCmp3: Int64
AcceptedCmp4: Int64
AcceptedCmp5: Int64
AcceptedCmp1: Int64
AcceptedCmp2: Int64
Complain: Int64
Z_CostContact: Int64
Z_Revenue: Int64
Response: Int64

NULL_COUNT:
shape: (1, 29)
┌─────┬────────────┬───────────┬──────────────┬───┬──────────┬──────────────┬───────────┬──────────┐
│ ID  ┆ Year_Birth ┆ Education ┆ Marital_Stat ┆ … ┆ Complain ┆ Z_CostContac ┆ Z_Revenue ┆ Response │
│ --- ┆ ---        ┆ ---       ┆ us           ┆   ┆ ---      ┆ t            ┆ ---       ┆ ---      │
│ u32 ┆ u32        ┆ u32       ┆ ---          ┆   ┆ u32      ┆ ---          ┆ u32       ┆ u32 

In [20]:
print("ROW_COUNT:", df.height)
print("UNIQUE_ID_COUNT:", df["ID"].n_unique())
print("DUPLICATE_ID_COUNT:", df.height - df["ID"].n_unique())

print("\nYEAR_BIRTH_MIN_MAX:")
print(df.select(
    pl.col("Year_Birth").min().alias("min_year"),
    pl.col("Year_Birth").max().alias("max_year")
))

print("\nINCOME_MIN_MAX:")
print(df.select(
    pl.col("Income").min().alias("min_income"),
    pl.col("Income").max().alias("max_income")
))

constant_cols = [col for col in df.columns if df[col].n_unique() == 1]
print("\nCONSTANT_COLUMNS:", constant_cols)

ROW_COUNT: 2240
UNIQUE_ID_COUNT: 2240
DUPLICATE_ID_COUNT: 0

YEAR_BIRTH_MIN_MAX:
shape: (1, 2)
┌──────────┬──────────┐
│ min_year ┆ max_year │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 1893     ┆ 1996     │
└──────────┴──────────┘

INCOME_MIN_MAX:
shape: (1, 2)
┌────────────┬────────────┐
│ min_income ┆ max_income │
│ ---        ┆ ---        │
│ i64        ┆ i64        │
╞════════════╪════════════╡
│ 1730       ┆ 666666     │
└────────────┴────────────┘

CONSTANT_COLUMNS: ['Z_CostContact', 'Z_Revenue']


In [21]:
suspicious_birth = df.filter((pl.col("Year_Birth") < 1900) | (pl.col("Year_Birth") > 2010))
suspicious_income = df.filter(pl.col("Income") > 200000)

print("SUSPICIOUS_BIRTH_COUNT:", suspicious_birth.height)
print(suspicious_birth.select(["ID", "Year_Birth"]).sort("Year_Birth"))

print("\nSUSPICIOUS_INCOME_COUNT:", suspicious_income.height)
print(suspicious_income.select(["ID", "Income"]).sort("Income", descending=True))

SUSPICIOUS_BIRTH_COUNT: 2
shape: (2, 2)
┌───────┬────────────┐
│ ID    ┆ Year_Birth │
│ ---   ┆ ---        │
│ i64   ┆ i64        │
╞═══════╪════════════╡
│ 11004 ┆ 1893       │
│ 1150  ┆ 1899       │
└───────┴────────────┘

SUSPICIOUS_INCOME_COUNT: 1
shape: (1, 2)
┌──────┬────────┐
│ ID   ┆ Income │
│ ---  ┆ ---    │
│ i64  ┆ i64    │
╞══════╪════════╡
│ 9432 ┆ 666666 │
└──────┴────────┘


In [22]:
suspicious_records = df.filter(
    pl.col("ID").is_in([11004, 1150, 9432])
).sort("ID")

print("SUSPICIOUS_RECORDS_COUNT:", suspicious_records.height)
print(suspicious_records)

SUSPICIOUS_RECORDS_COUNT: 3
shape: (3, 29)
┌───────┬────────────┬────────────┬─────────────┬───┬──────────┬────────────┬───────────┬──────────┐
│ ID    ┆ Year_Birth ┆ Education  ┆ Marital_Sta ┆ … ┆ Complain ┆ Z_CostCont ┆ Z_Revenue ┆ Response │
│ ---   ┆ ---        ┆ ---        ┆ tus         ┆   ┆ ---      ┆ act        ┆ ---       ┆ ---      │
│ i64   ┆ i64        ┆ str        ┆ ---         ┆   ┆ i64      ┆ ---        ┆ i64       ┆ i64      │
│       ┆            ┆            ┆ str         ┆   ┆          ┆ i64        ┆           ┆          │
╞═══════╪════════════╪════════════╪═════════════╪═══╪══════════╪════════════╪═══════════╪══════════╡
│ 1150  ┆ 1899       ┆ PhD        ┆ Together    ┆ … ┆ 0        ┆ 3          ┆ 11        ┆ 0        │
│ 9432  ┆ 1977       ┆ Graduation ┆ Together    ┆ … ┆ 0        ┆ 3          ┆ 11        ┆ 0        │
│ 11004 ┆ 1893       ┆ 2n Cycle   ┆ Single      ┆ … ┆ 0        ┆ 3          ┆ 11        ┆ 0        │
└───────┴────────────┴────────────┴─────────────

In [23]:
print("EDUCATION_VALUES:")
print(df.group_by("Education").len().sort("len", descending=True))

print("\nMARITAL_STATUS_VALUES:")
print(df.group_by("Marital_Status").len().sort("len", descending=True))

EDUCATION_VALUES:
shape: (5, 2)
┌────────────┬──────┐
│ Education  ┆ len  │
│ ---        ┆ ---  │
│ str        ┆ u32  │
╞════════════╪══════╡
│ Graduation ┆ 1127 │
│ PhD        ┆ 486  │
│ Master     ┆ 370  │
│ 2n Cycle   ┆ 203  │
│ Basic      ┆ 54   │
└────────────┴──────┘

MARITAL_STATUS_VALUES:
shape: (8, 2)
┌────────────────┬─────┐
│ Marital_Status ┆ len │
│ ---            ┆ --- │
│ str            ┆ u32 │
╞════════════════╪═════╡
│ Married        ┆ 864 │
│ Together       ┆ 580 │
│ Single         ┆ 480 │
│ Divorced       ┆ 232 │
│ Widow          ┆ 77  │
│ Alone          ┆ 3   │
│ Absurd         ┆ 2   │
│ YOLO           ┆ 2   │
└────────────────┴─────┘


In [24]:
weird_marital = df.filter(
    pl.col("Marital_Status").is_in(["Absurd", "YOLO", "Alone"])
).sort("Marital_Status")

print("WEIRD_MARITAL_COUNT:", weird_marital.height)
print(weird_marital)

WEIRD_MARITAL_COUNT: 7
shape: (7, 29)
┌───────┬────────────┬────────────┬─────────────┬───┬──────────┬────────────┬───────────┬──────────┐
│ ID    ┆ Year_Birth ┆ Education  ┆ Marital_Sta ┆ … ┆ Complain ┆ Z_CostCont ┆ Z_Revenue ┆ Response │
│ ---   ┆ ---        ┆ ---        ┆ tus         ┆   ┆ ---      ┆ act        ┆ ---       ┆ ---      │
│ i64   ┆ i64        ┆ str        ┆ ---         ┆   ┆ i64      ┆ ---        ┆ i64       ┆ i64      │
│       ┆            ┆            ┆ str         ┆   ┆          ┆ i64        ┆           ┆          │
╞═══════╪════════════╪════════════╪═════════════╪═══╪══════════╪════════════╪═══════════╪══════════╡
│ 7734  ┆ 1993       ┆ Graduation ┆ Absurd      ┆ … ┆ 0        ┆ 3          ┆ 11        ┆ 1        │
│ 4369  ┆ 1957       ┆ Master     ┆ Absurd      ┆ … ┆ 0        ┆ 3          ┆ 11        ┆ 0        │
│ 433   ┆ 1958       ┆ Master     ┆ Alone       ┆ … ┆ 0        ┆ 3          ┆ 11        ┆ 0        │
│ 7660  ┆ 1973       ┆ PhD        ┆ Alone       ┆ … ┆

In [25]:
df_qc = df.with_columns(
    [
        (pl.col("Year_Birth") < 1900).alias("flag_invalid_birth_year"),
        (pl.col("Income") > 200000).alias("flag_invalid_income"),
        pl.col("Marital_Status").is_in(["Absurd", "YOLO"]).alias("flag_invalid_marital_status"),
        pl.col("Marital_Status").eq("Alone").alias("flag_review_marital_status"),
    ]
).with_columns(
    (
        pl.col("flag_invalid_birth_year")
        | pl.col("flag_invalid_income")
        | pl.col("flag_invalid_marital_status")
        | pl.col("flag_review_marital_status")
    ).alias("flag_any_quality_issue")
)

print("DF_QC_SHAPE:", df_qc.shape)

print("\nFLAG_SUMMARY:")
print(
    df_qc.select(
        [
            pl.col("flag_invalid_birth_year").sum().alias("invalid_birth_year"),
            pl.col("flag_invalid_income").sum().alias("invalid_income"),
            pl.col("flag_invalid_marital_status").sum().alias("invalid_marital_status"),
            pl.col("flag_review_marital_status").sum().alias("review_marital_status"),
            pl.col("flag_any_quality_issue").sum().alias("any_quality_issue"),
        ]
    )
)

print("\nQC_SAMPLE:")
print(
    df_qc.filter(pl.col("flag_any_quality_issue"))
    .select(
        [
            "ID",
            "Year_Birth",
            "Income",
            "Marital_Status",
            "flag_invalid_birth_year",
            "flag_invalid_income",
            "flag_invalid_marital_status",
            "flag_review_marital_status",
        ]
    )
    .sort("ID")
)

DF_QC_SHAPE: (2240, 34)

FLAG_SUMMARY:
shape: (1, 5)
┌────────────────────┬────────────────┬────────────────────┬───────────────────┬───────────────────┐
│ invalid_birth_year ┆ invalid_income ┆ invalid_marital_st ┆ review_marital_st ┆ any_quality_issue │
│ ---                ┆ ---            ┆ atus               ┆ atus              ┆ ---               │
│ u32                ┆ u32            ┆ ---                ┆ ---               ┆ u32               │
│                    ┆                ┆ u32                ┆ u32               ┆                   │
╞════════════════════╪════════════════╪════════════════════╪═══════════════════╪═══════════════════╡
│ 2                  ┆ 1              ┆ 4                  ┆ 3                 ┆ 10                │
└────────────────────┴────────────────┴────────────────────┴───────────────────┴───────────────────┘

QC_SAMPLE:
shape: (10, 8)
┌───────┬────────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬────────────┐
│ ID    ┆ Y

In [31]:
print("INVALID_BIRTH_ROWS:", df_qc.filter(pl.col("flag_invalid_birth_year")).height)
print("INVALID_INCOME_ROWS:", df_qc.filter(pl.col("flag_invalid_income")).height)
print("INVALID_MARITAL_ROWS:", df_qc.filter(pl.col("flag_invalid_marital_status")).height)

to_remove = df_qc.filter(
    pl.col("flag_invalid_birth_year")
    | pl.col("flag_invalid_income")
    | pl.col("flag_invalid_marital_status")
)

print("\nROWS_TO_REMOVE_UNIQUE:", to_remove.height)
print(to_remove.select(["ID", "Year_Birth", "Income", "Marital_Status"]).sort("ID"))

INVALID_BIRTH_ROWS: 2
INVALID_INCOME_ROWS: 1
INVALID_MARITAL_ROWS: 4

ROWS_TO_REMOVE_UNIQUE: 7
shape: (7, 4)
┌───────┬────────────┬────────┬────────────────┐
│ ID    ┆ Year_Birth ┆ Income ┆ Marital_Status │
│ ---   ┆ ---        ┆ ---    ┆ ---            │
│ i64   ┆ i64        ┆ i64    ┆ str            │
╞═══════╪════════════╪════════╪════════════════╡
│ 492   ┆ 1973       ┆ 48432  ┆ YOLO           │
│ 1150  ┆ 1899       ┆ 83532  ┆ Together       │
│ 4369  ┆ 1957       ┆ 65487  ┆ Absurd         │
│ 7734  ┆ 1993       ┆ 79244  ┆ Absurd         │
│ 9432  ┆ 1977       ┆ 666666 ┆ Together       │
│ 11004 ┆ 1893       ┆ 60182  ┆ Single         │
│ 11133 ┆ 1973       ┆ 48432  ┆ YOLO           │
└───────┴────────────┴────────┴────────────────┘


In [29]:
ids_to_remove = (
    df_qc
    .filter(
        pl.col("flag_invalid_birth_year")
        | pl.col("flag_invalid_income")
        | pl.col("flag_invalid_marital_status")
    )
    .select("ID")
    .to_series()
    .to_list()
)

print("IDS_TO_REMOVE:", ids_to_remove)
print("IDS_TO_REMOVE_COUNT:", len(ids_to_remove))

df_clean = (
    df_qc
    .filter(~pl.col("ID").is_in(ids_to_remove))
    .drop(
        [
            "Z_CostContact",
            "Z_Revenue",
            "flag_invalid_birth_year",
            "flag_invalid_income",
            "flag_invalid_marital_status",
            "flag_review_marital_status",
            "flag_any_quality_issue",
        ]
    )
)

print("\nORIGINAL_ROWS:", df_qc.height)
print("CLEAN_ROWS:", df_clean.height)
print("ROWS_REMOVED:", df_qc.height - df_clean.height)
print("SHOULD_EQUAL_7:", (df_qc.height - df_clean.height) == 7)

IDS_TO_REMOVE: [11004, 1150, 7734, 4369, 492, 11133, 9432]
IDS_TO_REMOVE_COUNT: 7

ORIGINAL_ROWS: 2240
CLEAN_ROWS: 2233
ROWS_REMOVED: 7
SHOULD_EQUAL_7: True


In [32]:
df_clean.write_csv("cleaned_leads.csv")